# 🔍 Modul 03: Exploratory Data Analysis (EDA) & Feature Engineering
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/antonprafanto/data-mining-zero-to-hero/blob/main/notebooks/03_EDA_and_Feature_Engineering.ipynb)
[![Course](https://img.shields.io/badge/Course-Data%20Mining%20Zero%20to%20Hero-blue.svg)](https://github.com/antonprafanto/data-mining-zero-to-hero)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)

> **Pengampu**: Anton Prafanto  
> **Prinsip Pembelajaran**: *Intuition before Formulas* — Pahami cerita di balik data secara visual sebelum membangun model matematika rumit.

---

### 🗺️ Peta Posisi Modul dalam Siklus CRISP-DM
```
[1. Business Understanding]
            │
            ▼
🎯 [2. DATA UNDERSTANDING (EDA)] ◄── Anda berada di sini!
            │
            ▼
🎯 [3. DATA PREPARATION (FEATURE ENGINEERING)] ◄── Dan di sini!
            │
            ▼
[4. Modeling] ➔ [5. Evaluation] ➔ [6. Deployment]
```

Banyak pemula beranggapan bahwa kunci keberhasilan AI terletak pada algoritma yang paling rumit.  
Namun, guru besar AI dunia, **Prof. Andrew Ng (Stanford University)**, menegaskan:
> *"Coming up with features is difficult, time-consuming, requires expert knowledge. Applied Machine Learning is basically Feature Engineering."*

Di **Modul 03**, kita akan menguasai dua keahlian paling fundamental bagi praktisi data:
1. **Exploratory Data Analysis (EDA)**: Menjadi detektif data untuk menelusuri pola visual secara univariat, bivariat, dan multivariat.
2. **Feature Engineering**: Seni merekayasa fitur-fitur baru yang cerdas untuk melipatgandakan kecerdasan model machine learning!


## 🕵️ 1. Filosofi EDA: Analogi Dokter Forensik & Kamus Istilah Awam

### 🩺 Analogi Dokter Sebelum Membedah Pasien
Bayangkan seorang dokter spesialis bedah:
* Apakah dokter akan langsung membawa pasien ke ruang operasi dan membedahnya hanya karena pasien mengeluh pusing?  
  **Tentu tidak!** Tindakan gegabah itu bisa berakibat fatal.
* Dokter akan terlebih dahulu melakukan **pemeriksaan komprehensif**: mengukur tensi, cek darah lengkap, rontgen paru-paru, hingga USG/MRI.
* **EDA adalah stetoskop dan rontgen bagi praktisi data**: Kita dilarang keras melatih model machine learning sebelum memahami distribusi, korelasi, dan karakteristik yang tersimpan di dalam data!

---

### 📖 Kamus Istilah Awam (*Jargon Buster Modul 03*)
| Istilah Teknis | Bahasa Manusiawi / Analogi Sehari-hari |
| :--- | :--- |
| **Univariat** | Menganalisis **1 kolom sendirian** (seperti menimbang berat badan seseorang tanpa melihat tingginya). |
| **Bivariat** | Menganalisis hubungan antara **2 kolom** (seperti mengukur apakah orang yang lebih tinggi cenderung berbobot lebih berat). |
| **Multivariat** | Menganalisis interaksi **seluruh kolom sekaligus** dalam satu matriks visual komprehensif. |
| **Skewness** | Derajat kemiringan ekor grafik (apakah data menumpuk di kiri atau menumpuk di kanan). |
| **Kurtosis** | Derajat keruncingan puncak grafik (apakah puncaknya runcing tajam atau landai mendatar). |
| **Multikolinearitas** | Kondisi "kembar identik" di mana dua kolom prediktor membawa informasi yang sama persis sehingga membingungkan model. |
| **Curse of Dimensionality** | Bahaya membuat terlalu banyak kolom baru yang membuat ruang data menjadi kosong melompong (*sparse*) dan model overfit. |
| **Binning / Diskretisasi** | Memotong angka kontinu menjadi kotak-kotak kategori (seperti mengelompokkan umur ke: Anak, Remaja, Dewasa). |
| **Feature Importance** | Skor persentase pengaruh suatu fitur dalam membantu model menebak target secara akurat. |

---

### 📂 Dataset Studi Kasus: Transaksi Restoran (*Tips Dataset*)
Kita akan menggunakan dataset transaksi restoran nyata (*Tips Dataset*):
* `total_bill`: Jumlah tagihan makanan dalam Dollar ($).
* `tip`: Uang tip yang diberikan oleh pelanggan kepada pelayan ($).
* `sex`: Jenis kelamin pembayar (`Male` / `Female`).
* `smoker`: Apakah pelanggan duduk di area merokok (`Yes` / `No`).
* `day`: Hari kunjungan (`Thur`, `Fri`, `Sat`, `Sun`).
* `time`: Waktu santap (`Lunch` / `Dinner`).
* `size`: Jumlah orang dalam rombongan meja makan.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Atur konfigurasi visualisasi agar elegan dan seragam
sns.set_theme(style='whitegrid')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

# Memuat dataset transaksi restoran dengan fallback yang aman
try:
    df = sns.load_dataset('tips')
except Exception:
    url_fallback = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"
    df = pd.read_csv(url_fallback)

print(f"Dimensi Data: {df.shape[0]} baris transaksi × {df.shape[1]} kolom")
print("\n5 Baris Sampel Data:")
df.head()


## 📊 2. Alur EDA Bagian 1: Analisis Univariat (*One by One*)

**Analisis Univariat** adalah pemeriksaan setiap kolom data secara mandiri tanpa menghubungkannya dengan kolom lain:
* **Untuk Variabel Numerik Kontinu**: Kita mengamati bentuk sebaran, pemusatan (mean, median), kemiringan (**Skewness**), dan keruncingan (**Kurtosis**).
* **Untuk Variabel Kategorikal**: Kita mengamati frekuensi kemunculan tiap kategori menggunakan diagram batang (*Countplot*).

---

### 📐 Panduan Membaca Metrik Bentuk Distribusi:
| Metrik | Rentang Nilai | Makna & Interpretasi Awam |
| :--- | :--- | :--- |
| **Skewness** | **$> +1.0$** | **Sangat Miring ke Kanan (Right-Skewed)**: Ekor panjang ke kanan. Mayoritas data bernilai kecil, sedikit data bernilai raksasa (contoh: gaji warga, harga rumah, tagihan restoran). |
| | **$-0.5$ s/d $+0.5$** | **Simetris / Distribusi Normal**: Seimbang kiri dan kanan seperti lonceng simetris. |
| | **$< -1.0$** | **Sangat Miring ke Kiri (Left-Skewed)**: Ekor panjang ke kiri. Mayoritas data bernilai besar. |
| **Kurtosis** | **$> 3.0$** (atau excess $> 0$) | **Leptokurtik**: Puncak sangat runcing dan ekor tebal (*fat tails*), rawan outlier ekstrem. |
| | **$pprox 3.0$** (atau excess $pprox 0$) | **Mesokurtik**: Kelengkungan puncak normal lonceng Gaussian. |
| | **$< 3.0$** (atau excess $< 0$) | **Platikurtik**: Puncak mendatar dan landai. |


In [ ]:
# 1. Menghitung Skewness dan Kurtosis untuk Tagihan dan Tip
print("=== UKURAN BENTUK DISTRIBUSI (SHAPE METRICS) ===")
print(f"Skewness Total Bill : {df['total_bill'].skew():.2f} (Right-Skewed > 1.0)")
print(f"Kurtosis Total Bill : {df['total_bill'].kurt():.2f}")
print(f"Skewness Tip        : {df['tip'].skew():.2f} (Right-Skewed > 1.0)")
print(f"Kurtosis Tip        : {df['tip'].kurt():.2f}")

# 2. Visualisasi Univariat Terpadu: Histogram + KDE, Boxplot, dan Countplot
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Subplot 1: Histogram + Kurva Densitas (KDE)
sns.histplot(df['total_bill'], kde=True, color='teal', ax=axes[0])
axes[0].set_title("Distribusi Total Tagihan ($)", fontweight='bold')
axes[0].set_xlabel("Total Tagihan ($)")

# Subplot 2: Boxplot Univariat (Melihat Kuartil & Outlier)
sns.boxplot(y=df['total_bill'], color='lightseagreen', ax=axes[1])
axes[1].set_title("Boxplot Total Tagihan ($)", fontweight='bold')
axes[1].set_ylabel("Total Tagihan ($)")

# Subplot 3: Countplot Frekuensi Hari Transaksi
sns.countplot(data=df, x='day', hue='day', palette='Set2', legend=False, ax=axes[2])
axes[2].set_title("Frekuensi Transaksi per Hari", fontweight='bold')
axes[2].set_xlabel("Hari Kunjungan")
axes[2].set_ylabel("Jumlah Transaksi")

plt.tight_layout()
plt.show()

print("💡 Wawasan Univariat:")
print("1. Total tagihan miring ke kanan (Right-skewed): Sebagian besar tamu belanja $10-$20, namun ada tamu yang belanja di atas $40.")
print("2. Hari Sabtu dan Minggu adalah hari paling ramai transaksi.")


## 📈 3. Alur EDA Bagian 2: Analisis Bivariat (*Relationship Between 2 Variables*)

Di tahap ini, detektif data mulai menyelidiki hubungan kausalitas: *"Apakah variabel A memengaruhi variabel B?"*

---

### A. Numerik vs Numerik: Korelasi Pearson vs Spearman
* **Korelasi Pearson ($r$)**: Mengukur kekuatan hubungan **garis lurus linear sempurna**. Sangat sensitif terhadap outlier ekstrem.
* **Korelasi Spearman ($\rho$)**: Mengukur kekuatan hubungan **berbasis urutan peringkat (*monotonic*)**. Tangguh terhadap data yang tidak berdistribusi normal atau memiliki outlier!

---

### B. Numerik vs Kategorikal: Perbandingan Kelompok (Boxplot)
Apakah tamu pria dan wanita memesan tagihan yang berbeda tergantung harinya?

---

### C. Kategorikal vs Kategorikal: Tabel Kontinjensi (*Cross-Tabulation*)
Apakah tamu yang datang di hari kerja dominan makan siang (*Lunch*) atau makan malam (*Dinner*)?


In [ ]:
# 1. Menghitung Korelasi Pearson vs Spearman
corr_pearson = df['total_bill'].corr(df['tip'], method='pearson')
corr_spearman = df['total_bill'].corr(df['tip'], method='spearman')

print("=== KORELASI ANTARA TOTAL TAGIHAN & BESAR TIP ===")
print(f"Korelasi Pearson  (Linear)    : {corr_pearson:.3f}")
print(f"Korelasi Spearman (Peringkat) : {corr_spearman:.3f}")

# 2. Tabel Kontinjensi (Cross-Tabulation) dengan Persentase
print("\n=== TABEL KONTINJENSI (CROSSTAB): HARI VS WAKTU MAKAN ===")
tabel_silang = pd.crosstab(df['day'], df['time'], margins=True)
print(tabel_silang)

persentase_waktu = pd.crosstab(df['day'], df['time'], normalize='index') * 100
print("\nPersentase Waktu Makan per Hari (%):")
print(persentase_waktu.round(1))

# 3. Visualisasi Bivariat: Scatter Plot dengan Garis Tren Regresi + Boxplot Komparatif
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Scatter plot dengan garis tren linear (regplot)
sns.regplot(
    data=df, x='total_bill', y='tip',
    scatter_kws={'alpha': 0.6, 'color': 'darkslateblue'},
    line_kws={'color': 'crimson', 'lw': 2},
    ax=axes[0]
)
axes[0].set_title(f"Hubungan Tagihan vs Tip (Pearson r = {corr_pearson:.2f})", fontweight='bold')
axes[0].set_xlabel("Total Tagihan ($)")
axes[0].set_ylabel("Uang Tip ($)")

# Boxplot Komparasi Hari vs Tagihan berdasarkan Gender
sns.boxplot(data=df, x='day', y='total_bill', hue='sex', palette='Pastel1', ax=axes[1])
axes[1].set_title("Komparasi Total Tagihan per Hari Berdasarkan Gender", fontweight='bold')
axes[1].set_xlabel("Hari")
axes[1].set_ylabel("Total Tagihan ($)")

plt.tight_layout()
plt.show()

print("💡 Wawasan Bivariat:")
print("1. Ada korelasi positif kuat (+0.68): Semakin mahal tagihan makan, semakin besar tip yang ditinggalkan tamu.")
print("2. Di akhir pekan (Sabtu & Minggu), restoran HANYA menyajikan makan malam (Dinner = 100%), tidak ada santap siang sama sekali!")


## 🌐 4. Alur EDA Bagian 3: Analisis Multivariat (*All Variables at Once*)

Ketika kita ingin melihat gambaran besar dari seluruh variabel sekaligus:
1. **Pairplot (`sns.pairplot`)**: Matriks visualisasi paling sakti dalam data mining! Menampilkan scatter plot untuk seluruh kombinasi pasangan kolom angka secara serentak, diwarnai oleh target kategori (`hue`).
2. **Heatmap Korelasi Numerik**: Mengukur korelasi antar seluruh kolom numerik ke dalam matriks berwarna.

---

### ⚠️ Bahaya Multikolinearitas (*Multicollinearity*)
Jika di dalam dataset terdapat dua fitur independen yang saling berkorelasi sangat tinggi ($r > 0.85$ atau $r > 0.90$):
* *Contoh*: Kolom `luas_meter_persegi` dan `luas_kaki_persegi`.
* **Bahaya**: Kedua fitur membawa informasi yang redundan (sama persis), yang dapat merusak estimasi bobot pada model linear. Salah satu dari pasangan tersebut wajib dieliminasi!


In [ ]:
# 1. Pairplot Matriks Sebaran Multivariat
pairplot_fig = sns.pairplot(df, hue='time', palette='Dark2', diag_kind='kde')
pairplot_fig.fig.suptitle('Pairplot Multivariat Dataset Tips Restoran (Warna: Waktu Makan)', y=1.02, fontweight='bold')
plt.show()

# 2. Heatmap Matriks Korelasi Numerik
plt.figure(figsize=(6, 4.5))
kolom_numerik = df.select_dtypes(include=[np.number])
korelasi_matriks = kolom_numerik.corr()

sns.heatmap(
    korelasi_matriks,
    annot=True,
    cmap='coolwarm',
    fmt='.2f',
    vmin=-1,
    vmax=1,
    linewidths=0.5
)
plt.title("Heatmap Korelasi Seluruh Fitur Numerik", fontweight='bold', pad=12)
plt.show()

print("💡 Cara Membaca Heatmap:")
print("- Angka diagonal bernilai 1.00 (korelasi suatu variabel dengan dirinya sendiri).")
print("- Korelasi tertinggi ada pada pasangan total_bill dan tip (0.68).")
print("- Tidak ada multikolinearitas ekstrem (> 0.85), sehingga seluruh fitur aman dipertahankan.")


## 🛠️ 5. Seni Rekayasa Fitur (*Feature Engineering*)

Data mentah jarang sekali siap pakai untuk menghasilkan model AI berakurasi tinggi. Melalui **Feature Engineering**, kita menciptakan fitur baru yang membawa sinyal bisnis berharga:

---

### A. Teknik 1: Ekstraksi Fitur Tanggal & Waktu (`datetime`)
Format teks tanggal seperti `"2026-01-01 10:00:00"` tidak bisa dihitung langsung oleh komputer. Namun jika kita pecah:
* `jam`: 10 (Pagi) / 19 (Malam)
* `hari_ke`: Thursday
* `kuartal`: Kuartal 1 s/d 4 (Q1, Q2, Q3, Q4)
* `is_weekend`: 0 (Hari kerja) / 1 (Akhir pekan)
* Fitur-fitur pecahan ini memiliki korelasi kuat terhadap pola perilaku belanja pelanggan!

---

### B. Teknik 2: Diskretisasi Nilai Kontinu (*Binning* via `pd.cut` & `pd.qcut`)
Mengubah angka kontinu menjadi kategori sering kali menstabilkan model:
* **`pd.cut` (Equal Width / Nilai Sama Lebar)**: Membagi data berdasarkan batas interval nilai tertentu. Jumlah sampel per bin bisa berbeda-beda.
* **`pd.qcut` (Equal Frequency / Kuantil Sama Banyak)**: Membagi data sedemikian rupa sehingga setiap kelompok memiliki jumlah baris data yang **sama banyak** (misal: tepat 33.3% per bin).

---

### C. Teknik 3: Pembuatan Fitur Rasio & Normalisasi Pengeluaran
Dua pelanggan sama-sama memberi tip $5:
* Pelanggan A memesan tagihan $10 (Tip $5 = **50% dari tagihan** ➔ Sangat Dermawan!).
* Pelanggan B memesan tagihan $100 (Tip $5 = **5% dari tagihan** ➔ Sangat Pelit!).
Fitur rasio `tip_pct` dan `bill_per_person` menangkap konteks nyata ini!


In [ ]:
# A. Ekstraksi Fitur Datetime Lengkap (Simulasi Transaksi Setahun Penuh)
waktu_simulasi = pd.date_range(start='2026-01-01 10:00:00', periods=len(df), freq='36h')
df['waktu_transaksi'] = waktu_simulasi

df['jam'] = df['waktu_transaksi'].dt.hour
df['hari_ke'] = df['waktu_transaksi'].dt.day_name()
df['kuartal'] = df['waktu_transaksi'].dt.quarter
df['is_weekend'] = df['waktu_transaksi'].dt.dayofweek.apply(lambda x: 1 if x >= 5 else 0)

print("Contoh Fitur Datetime yang Berhasil Diekstrak (Menjangkau Q1 - Q4):")
print(df[['waktu_transaksi', 'jam', 'hari_ke', 'kuartal', 'is_weekend']].head(3))

# B. Diskretisasi / Binning: Perbandingan Nyata pd.cut vs pd.qcut
# 1. pd.cut (Batas Interval Tetap: Hemat < $15, Menengah $15-$30, Mewah > $30)
df['kategori_cut'] = pd.cut(df['total_bill'], bins=[0, 15, 30, 100], labels=['Hemat', 'Menengah', 'Mewah'])

# 2. pd.qcut (Kuantil 3 Bagian: Jumlah Data Tiap Kategori Sama Banyak ~ 33.3%)
df['kategori_qcut'] = pd.qcut(df['total_bill'], q=3, labels=['Hemat', 'Menengah', 'Mewah'])

print("\n=== PERBANDINGAN DISTRIBUSI BINNING ===")
print("Distribusi pd.cut (Batas Interval Tetap):")
print(df['kategori_cut'].value_counts())
print("\nDistribusi pd.qcut (Jumlah Baris Sama Rata):")
print(df['kategori_qcut'].value_counts())

# C. Pembuatan Fitur Rasio
df['tip_pct'] = (df['tip'] / df['total_bill']) * 100
df['bill_per_person'] = df['total_bill'] / df['size']

print("\n=== CONTOH FITUR RASIO BARU ===")
df[['total_bill', 'tip', 'size', 'tip_pct', 'bill_per_person']].head(4)


## 🎯 6. Seleksi Fitur & Bahaya *Curse of Dimensionality*

### 🌪️ Apa Itu *Curse of Dimensionality* (Bencana Dimensi)?
Ada mitos di kalangan pemula: *"Semakin banyak kolom/fitur yang kita buat, model pasti makin pintar!"*.  
**SALAH BESAR!**  
1. **Ruang Data Menjadi Renggang (*Sparse Space*)**: Menambah dimensi membuat volume ruang geometris meledak secara eksponensial. Titik-titik data menjadi terisolasi saling berjauhan, sehingga komputer membutuhkan jutaan data tambahan hanya untuk mengisi kekosongan ruang tersebut.
2. **Bahaya Overfitting**: Model mulai menghafal derau/noise acak bukannya belajar pola asli.
3. **Komputasi Berat**: Waktu komputasi pelatihan model melambat drastis.

---

### 🗂️ 3 Keluarga Metode Seleksi Fitur (Analogi Seleksi Tim Juara):
1. **Filter Methods (Seleksi Berkas Administrasi)**:  
   * *Analogi*: Menyaring pelamar kerja hanya berdasarkan syarat IPK minimal $\ge 3.0$ tanpa melihat keahlian lainnya.  
   * *Teknik*: Menghitung skor korelasi atau variansi statistik secara independen sebelum melatih model. Sangat cepat dan hemat komputasi.
2. **Wrapper Methods (Audisi Eliminasi Bertahap)**:  
   * *Analogi*: Seperti audisi menyanyi di televisi di mana peserta diuji dalam berbagai kombinasi duet/grup, dan yang performanya paling buruk dieliminasi satu per satu (*Recursive Feature Elimination / RFE*).  
   * *Teknik*: Sangat akurat, namun boros waktu komputasi karena melatih model berulang-ulang.
3. **Embedded Methods (Observasi Pelatih Selama Tanding)**:  
   * *Analogi*: Pelatih sepak bola yang mengamati kontribusi nyata pemain langsung di atas lapangan selama pertandingan berlangsung.  
   * *Teknik*: Seleksi fitur terintegrasi langsung saat algoritma melatih dirinya (contoh: **Feature Importance** pada Random Forest).


In [ ]:
from sklearn.ensemble import RandomForestRegressor

# 1. Penerapan Filter Method: Memeriksa korelasi absolut terhadap target 'tip'
fitur_numerik_semua = ['total_bill', 'size', 'jam', 'kuartal', 'is_weekend', 'bill_per_person', 'tip_pct']
korelasi_terhadap_tip = df[fitur_numerik_semua + ['tip']].corr()['tip'].drop('tip').abs().sort_values(ascending=False)

print("=== FILTER METHOD: KORELASI ABSOLUT TERHADAP TARGET 'TIP' ===")
print(korelasi_terhadap_tip.round(3).to_string())

# 2. Penerapan Embedded Method: Feature Importance Random Forest
fitur_kandidat = ['total_bill', 'size', 'jam', 'kuartal', 'is_weekend', 'bill_per_person']
X = df[fitur_kandidat]
y = df['tip']

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X, y)

skor_fitur = pd.Series(rf.feature_importances_, index=fitur_kandidat).sort_values(ascending=True)

# Visualisasi Bobot Pengaruh Fitur
plt.figure(figsize=(8, 4))
skor_fitur.plot(kind='barh', color='teal', edgecolor='black')
plt.title('Skor Kepentingan Fitur Terhadap Besar Tip (Feature Importance)', fontweight='bold')
plt.xlabel('Tingkat Pengaruh (Importance Score)')
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

print("\n=== EMBEDDED METHOD: PERINGKAT RANDOM FOREST IMPORTANCE ===")
print(skor_fitur.sort_values(ascending=False).round(4).to_string())

print("\n💡 Kesimpulan Seleksi Fitur:")
print(f"👉 Fitur '{skor_fitur.idxmax()}' dan 'bill_per_person' menyumbang lebih dari 85% pengaruh!")
print("Fitur seperti 'is_weekend' dan 'kuartal' memiliki pengaruh sangat kecil (< 3%) sehingga aman dieliminasi untuk meringankan model.")


## 🧠 7. Kuis Evaluasi Pemahaman Mandiri

Uji pemahaman Anda terhadap konsep EDA dan rekayasa fitur di Modul 03. Jalankan sel kode di bawah untuk memeriksa kunci jawabannya:

---
* **Soal 1**: Apa perbedaan mendasar antara analisis Univariat, Bivariat, dan Multivariat?
* **Soal 2**: Mengapa korelasi Spearman sering kali lebih diandalkan daripada korelasi Pearson saat berhadapan dengan data yang memiliki outlier ekstrem?
* **Soal 3**: Apa perbedaan konkret antara fungsi `pd.cut` dan `pd.qcut` pada Pandas?
* **Soal 4**: Apa yang dimaksud dengan Multikolinearitas dan mengapa fitur dengan korelasi di atas 0.85 berbahaya jika dipertahankan bersamaan?
* **Soal 5**: Mengapa fenomena *Curse of Dimensionality* menjadi alasan kuat kita wajib melakukan seleksi fitur?


In [ ]:
# Kunci jawaban logis evaluasi mandiri Modul 3
kunci_jawaban_modul_3 = {
    "Soal 1": "Univariat menganalisis 1 kolom secara terpisah (distribusi/skewness). Bivariat menganalisis hubungan antar 2 kolom (korelasi/crosstab). Multivariat menganalisis interaksi seluruh kolom secara serentak (pairplot/heatmap).",
    "Soal 2": "Korelasi Spearman mengukur hubungan berbasis urutan peringkat (rank-order) sehingga tidak terdistorsi oleh angka pencilan ekstrem, sedangkan Pearson mengukur garis lurus yang sangat sensitif terhadap outlier.",
    "Soal 3": "pd.cut membagi data berdasarkan batas interval nilai yang sama lebar (jumlah sampel per kelompok bisa berbeda). pd.qcut membagi data berdasarkan kuantil/persentase sehingga jumlah sampel per kelompok sama banyak.",
    "Soal 4": "Multikolinearitas terjadi ketika dua fitur independen saling berkorelasi sangat tinggi (> 0.85) sehingga membawa informasi redundan yang merusak kestabilan estimasi bobot model machine learning.",
    "Soal 5": "Menambah terlalu banyak fitur membuat ruang data menjadi renggang (sparse), meningkatkan risiko overfitting, dan memperlambat komputasi secara eksponensial."
}

print("=== KUNCI JAWABAN EVALUASI MANDIRI MODUL 03 ===")
for soal, pembahasan in kunci_jawaban_modul_3.items():
    print(f"👉 {soal}:\n   {pembahasan}\n")


## ✍️ 8. Kotak Latihan Bebas (*Playground Challenge*)

### 🎯 Misi Eksperimen: Siapa yang Paling Dermawan Memberi Tip?
Selesaikan investigasi analitik berikut:
1. Dari tabel transaksi, cari tahu **hari apa (`day`) yang memberikan rata-rata persentase tip (`tip_pct`) paling tinggi**!
2. Buat fitur biner baru bernama `rombongan_besar`: Bernilai `1` jika jumlah tamu (`size`) $> 3$, dan bernilai `0` jika `size` $\le 3$.
3. **Tantangan Investigasi**: Bandingkan rata-rata persentase tip antara rombongan besar vs rombongan kecil. Apakah meja yang diduduki banyak orang memberi persen tip lebih banyak atau lebih sedikit?


In [ ]:
# Solusi Tantangan Mandiri
# 1. Rata-rata persentase tip berdasarkan hari
rata_tip_hari = df.groupby('day', observed=False)['tip_pct'].mean().sort_values(ascending=False)

# 2. Rekayasa fitur biner rombongan_besar
df['rombongan_besar'] = df['size'].apply(lambda x: 1 if x > 3 else 0)

# 3. Analisis Investigasi: Rombongan Besar vs Kecil
tip_per_rombongan = df.groupby('rombongan_besar', observed=False)['tip_pct'].mean()

print("=== RATA-RATA PERSENTASE TIP BERDASARKAN HARI ===")
print(rata_tip_hari.round(2))
print(f"\n💵 Hari Paling Dermawan Memberi Tip: {rata_tip_hari.index[0]} ({rata_tip_hari.iloc[0]:.2f}%)")

print("\n=== DISTRIBUSI ROMBONGAN BESAR VS KECIL ===")
print(df['rombongan_besar'].value_counts())

print("\n=== PERBANDINGAN PERSEN TIP ROMBONGAN ===")
print(f"Rombongan Kecil (<= 3 orang): {tip_per_rombongan[0]:.2f}% tip")
print(f"Rombongan Besar (> 3 orang) : {tip_per_rombongan[1]:.2f}% tip")
print("👉 Temuan Bisnis: Rombongan kecil cenderung memberikan persentase tip yang lebih royal!")


## 🎯 9. Checklist Kelulusan Modul 03 (*Hero Checkpoint*)

Beri tanda centang pada hati Anda jika telah menguasai indikator penting berikut:
- [ ] Saya memahami posisi EDA (Data Understanding) dan Feature Engineering (Data Preparation) dalam alur CRISP-DM.
- [ ] Saya bisa melakukan Analisis Univariat: mengukur Skewness, Kurtosis, serta membaca grafik Histogram & Boxplot.
- [ ] Saya memahami perbedaan Korelasi Linear Pearson dan Korelasi Peringkat Spearman.
- [ ] Saya mampu membuat tabulasi silang kategori (*Cross-Tabulation*) dengan `pd.crosstab()`.
- [ ] Saya bisa membaca matriks visual multivariat dengan `sns.pairplot` dan Heatmap korelasi.
- [ ] Saya mampu mendeteksi bahaya Multikolinearitas ($r > 0.85$).
- [ ] Saya menguasai teknik ekstraksi tanggal/waktu (`.dt.hour`, `.dt.day_name()`, `.dt.quarter`, `.dt.dayofweek`).
- [ ] Saya memahami perbedaan konkret antara `pd.cut()` (lebar nilai) dan `pd.qcut()` (kuantil rata).
- [ ] Saya mampu merekayasa fitur rasio baru yang bermakna bisnis (`tip_pct`, `bill_per_person`).
- [ ] Saya memahami bahaya *Curse of Dimensionality* dan mampu menggunakan metode seleksi fitur (Filter & Embedded Random Forest).

---

## 🏁 Gerbang Menuju Level 2: Algoritma Klasifikasi!

Selamat! Anda kini telah menguasai seni menganalisis data dan merekayasa fitur-fitur baru yang cerdas.

Sekarang, saatnya memasuki **inti terdalam dari Data Mining: Melatih Algoritma Machine Learning**!

Mari kita mulai dari ranah **Supervised Learning - Klasifikasi (K-NN, Naive Bayes, Decision Tree, Random Forest)**:

👉 **[Buka Modul 04: Supervised Learning I - Klasifikasi](04_Classification_Algorithms.ipynb)**  
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/antonprafanto/data-mining-zero-to-hero/blob/main/notebooks/04_Classification_Algorithms.ipynb)

Sampai jumpa di Modul 04! Tetap konsisten dan nikmati setiap proses belajarnya! 🚀
